In [ ]:
from pathlib import Path
import sys
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
DATA_DIR = DATA_ROOT / "00_Shared_Data_and_Code/Data"
OUTPUT_DIR = PROJECT_ROOT / "01_Clinical_and_Cohort/02_Clinical_Model"
MODEL_SET = "C"

sys.path.insert(0, str(PROJECT_ROOT))
from modeling_pipeline import (
    load_dataset, model_estimators, training_search_grids,
    fit_on_training, positive_probability, model_metrics,
)

In [ ]:
X, y, split = load_dataset(DATA_DIR, MODEL_SET)
display(pd.crosstab(split, y, margins=True))
assert set(X.index[split == 'Train']).isdisjoint(X.index[split == 'Test'])

In [ ]:
train = split == 'Train'
X_train, y_train = X.loc[train], y.loc[train]
X_test, y_test = X.loc[~train], y.loc[~train]
display(pd.DataFrame({'N': [len(y_train), len(y_test)],
                      'HRD': [int(y_train.sum()), int(y_test.sum())]},
                     index=['Training', 'Test']))

In [ ]:
models = model_estimators(MODEL_SET)
search_grids = training_search_grids(MODEL_SET, models)

In [ ]:
fitted_models = {}
cv_tables = []
parameter_tables = {}
for name, classifier in models.items():
    if name not in search_grids:
        raise ValueError(f'Missing pre-specified training-CV search grid for {name}.')
    fitted, cv_result, parameters = fit_on_training(
        X_train, y_train, MODEL_SET, classifier, search_grids[name])
    fitted_models[name] = fitted
    cv_result['Model'] = name
    cv_tables.append(cv_result)
    parameter_tables[name] = parameters
cv = pd.concat(cv_tables, ignore_index=True)
display(cv)

In [ ]:
metric_rows = []
score_tables = []
for name, fitted in fitted_models.items():
    for part, features, target in [('Train', X_train, y_train), ('Test', X_test, y_test)]:
        probability = positive_probability(fitted, features)
        metric_rows.append(dict(Model=name, Split=part, **model_metrics(target, probability)))
        score_tables.append(pd.DataFrame({'ID': features.index, 'HRR_ANY': target.to_numpy(),
                                         'HRD_probability': probability, 'Model': name, 'Split': part}))
metrics = pd.DataFrame(metric_rows)
scores = pd.concat(score_tables, ignore_index=True)
display(metrics)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for (name, part), table in scores.groupby(['Model', 'Split'], sort=False):
    table.to_csv(OUTPUT_DIR / f'{MODEL_SET}_{name}_{part}_scores.csv', index=False)
for name, fitted in fitted_models.items():
    joblib.dump(fitted, OUTPUT_DIR / f'{MODEL_SET}_{name}_pipeline.joblib')
    feature_names = fitted['features'].get_feature_names_out()
    pd.DataFrame({'Feature': feature_names}).to_csv(
        OUTPUT_DIR / f'{MODEL_SET}_{name}_selected_features.csv', index=False)
    feature_values = pd.DataFrame(fitted['features'].transform(X), index=X.index, columns=feature_names)
    feature_values.insert(0, 'Split', split)
    feature_values.reset_index().to_csv(OUTPUT_DIR / f'{MODEL_SET}_{name}_feature_values.csv', index=False)
    (OUTPUT_DIR / f'{MODEL_SET}_{name}_training_parameters.json').write_text(
        json.dumps(parameter_tables[name], indent=2, ensure_ascii=False), encoding='utf8')
metrics.to_csv(OUTPUT_DIR / f'{MODEL_SET}_metrics.csv', index=False)
cv.to_csv(OUTPUT_DIR / f'{MODEL_SET}_training_cv.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=cv, x='Model', y='AUC', order=list(models), ax=ax)
ax.set(xlabel='Model', ylabel='Training cross-validation AUC', ylim=(0, 1))
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{MODEL_SET}_training_cv_AUC.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for model_name in models:
    fig, ax = plt.subplots(figsize=(4, 4))
    for part, color in [('Train', '#2474A8'), ('Test', '#D85F3B')]:
        current = scores[(scores.Model == model_name) & (scores.Split == part)]
        fpr, tpr, _ = roc_curve(current.HRR_ANY, current.HRD_probability, pos_label=1)
        auc = roc_auc_score(current.HRR_ANY, current.HRD_probability)
        ax.plot(fpr, tpr, color=color, label=f'{part}: AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=0.8)
    ax.set(xlabel='1 - Specificity', ylabel='Sensitivity', title=model_name)
    ax.legend(loc='lower right', fontsize=8)
    fig.savefig(OUTPUT_DIR / f'{MODEL_SET}_{model_name}_ROC.pdf', bbox_inches='tight')
    plt.close(fig)